# Document Processing Prototype

This notebook provides an incremental approach to testing the document processing pipeline for construction specifications.

## Overview

The notebook is structured to allow step-by-step development and validation:
1. **Environment Setup** - Install dependencies and configure paths
2. **Configuration** - Set up Docling pipeline options
3. **File Loading** - Load and inspect PDF files from the data directory
4. **Document Parsing** - Extract content using Docling
5. **Results Visualization** - Display and export parsed results
6. **Future Phases** - Placeholders for CSI classification, fact extraction, etc.



In [1]:
# Cell 1: Environment Setup and Dependencies

# Install required packages (run this cell first)
# Uncomment the following lines to install dependencies
# !pip install docling docling-core pymupdf pillow pandas numpy matplotlib IPython

import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional

# Configure logging for debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set up paths
PROJECT_ROOT = Path.cwd().parent  # Go up one level from notebooks/
DATA_DIR = PROJECT_ROOT / "data"

# List available PDF files
pdf_files = list(DATA_DIR.glob("*.pdf"))
print(f"\nFound {len(pdf_files)} PDF files:")
for pdf_file in pdf_files:
    print(f"  - {pdf_file.name}")



Found 3 PDF files:
  - Submittal and Product Description_redacted.pdf
  - Architectural Drawings_redacted.pdf
  - Spec 14 24 00 - Hydraulic Elevators.pdf


### Document Processing Configuration

In [8]:
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption
from docling.datamodel.pipeline_options import EasyOcrOptions


# Configure Docling pipeline options for construction documents
def create_docling_config():
    """Create Docling configuration optimized for construction documents"""
    
    # PDF pipeline options based on real construction document analysis
    pdf_options = PdfPipelineOptions(
        do_table_structure=True,
        do_ocr=True,
        ocr_options=EasyOcrOptions(
            lang=["en"],  # Specify your language(s)
            confidence_threshold=0.7,  # Higher threshold for better quality
            use_gpu=True,  # Enable GPU acceleration if available
            recog_network="standard"  # Use standard recognition network
        ),
        # RapidOcrOptions(
        #     backend="onnxruntime",  # Fast inference backend
        #     text_score=0.6,  # Higher confidence threshold
        #     force_full_page_ocr=True
        # ),
        # TesseractOcrOptions(
        #     lang=["eng"],
        #     psm=6  # Uniform block of text (good for most documents)
        # )
        images_scale=2.0,  # Higher scale for better text recognition
        force_backend_text=False  # Let OCR generate the text
    )
    
    # Format options
    format_options = {
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
    }
    
    return format_options

# Initialize document converter
try:
    format_options = create_docling_config()
    converter = DocumentConverter(format_options=format_options)
    print("✅ Docling document converter initialized successfully")
    print(f"   - Max page dimensions: 4000x4000 points")
    print(f"   - Annotation extraction: Enabled")
    print(f"   - XObject extraction: Enabled")
    print(f"   - OCR: Enabled")
    print(f"   - Table extraction: Enabled")
except Exception as e:
    print(f"❌ Error initializing Docling converter: {e}")
    print("Make sure to install docling and docling-core packages")
    converter = None


✅ Docling document converter initialized successfully
   - Max page dimensions: 4000x4000 points
   - Annotation extraction: Enabled
   - XObject extraction: Enabled
   - OCR: Enabled
   - Table extraction: Enabled


### File System Loading and Metadata Extraction

In [6]:
import fitz  # PyMuPDF for basic PDF metadata
from pathlib import Path

def get_pdf_metadata(pdf_path: Path) -> Dict[str, Any]:
    """Extract basic metadata from PDF file"""
    try:
        doc = fitz.open(pdf_path)
        metadata = {
            "filename": pdf_path.name,
            "file_size_bytes": pdf_path.stat().st_size,
            "page_count": len(doc),
            "pdf_version": doc.metadata.get("format", "Unknown"),
            "creator": doc.metadata.get("creator", "Unknown"),
            "producer": doc.metadata.get("producer", "Unknown"),
            "creation_date": doc.metadata.get("creationDate", ""),
            "modification_date": doc.metadata.get("modDate", ""),
            "title": doc.metadata.get("title", ""),
            "subject": doc.metadata.get("subject", ""),
        }
        
        # Get first page dimensions for drawing documents
        if len(doc) > 0:
            page = doc[0]
            rect = page.rect
            metadata["first_page_dimensions"] = {
                "width": rect.width,
                "height": rect.height,
                "rotation": page.rotation
            }
            
            # Count annotations on first page
            try:
                annotations = page.annots()
                metadata["first_page_annotations"] = len(annotations)
            except Exception as e:
                logger.warning(f"Could not get annotations for {pdf_path}: {e}")
                metadata["first_page_annotations"] = 0
        
        doc.close()
        return metadata
    except Exception as e:
        logger.error(f"Error reading metadata for {pdf_path}: {e}")
        return {"filename": pdf_path.name, "error": str(e)}

def display_document_info(pdf_files: List[Path]) -> Dict[str, Dict[str, Any]]:
    """Display information about all PDF files"""
    print("📄 Document Information")
    print("=" * 80)
    
    document_info = {}
    
    for pdf_file in pdf_files:
        print(f"\n📋 {pdf_file.name}")
        print("-" * 60)
        
        metadata = get_pdf_metadata(pdf_file)
        document_info[pdf_file.name] = metadata
        
        if "error" in metadata:
            print(f"❌ Error: {metadata['error']}")
            continue
            
        # File information
        size_mb = metadata["file_size_bytes"] / (1024 * 1024)
        print(f"   File size: {size_mb:.2f} MB")
        print(f"   Pages: {metadata['page_count']}")
        print(f"   PDF version: {metadata['pdf_version']}")
        
        # Creator information
        print(f"   Creator: {metadata['creator']}")
        print(f"   Producer: {metadata['producer']}")
        
        # Document type inference
        doc_type = infer_document_type(pdf_file.name, metadata)
        print(f"   Document type: {doc_type}")
        
        # Page information for drawings
        if "first_page_dimensions" in metadata:
            dims = metadata["first_page_dimensions"]
            print(f"   First page: {dims['width']:.0f} x {dims['height']:.0f} points")
            if dims["rotation"] != 0:
                print(f"   Page rotation: {dims['rotation']}°")
            print(f"   Annotations on first page: {metadata.get('first_page_annotations', 0)}")
    
    return document_info

def infer_document_type(filename: str, metadata: Dict[str, Any]) -> str:
    """Infer document type from filename and metadata"""
    filename_lower = filename.lower()
    
    if "spec" in filename_lower and any(div in filename for div in ["14 24", "07 21", "03 30"]):
        return "CSI Specification"
    elif "drawing" in filename_lower or "plan" in filename_lower:
        return "Architectural Drawing"
    elif "sd" in filename_lower or "pd" in filename_lower:
        return "Submittal Document"
    else:
        return "Construction Document"

# Load and display information about all PDF files
if pdf_files:
    document_info = display_document_info(pdf_files)
    print(f"\n✅ Loaded information for {len(document_info)} documents")
else:
    print("❌ No PDF files found in data directory")
    document_info = {}


2025-10-16 15:44:59,506 - __main__ - WARNING - Could not get annotations for /Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/data/Submittal and Product Description_redacted.pdf: object of type 'generator' has no len()
2025-10-16 15:44:59,511 - __main__ - WARNING - Could not get annotations for /Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/data/Architectural Drawings_redacted.pdf: object of type 'generator' has no len()
2025-10-16 15:44:59,513 - __main__ - WARNING - Could not get annotations for /Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/data/Spec 14 24 00 - Hydraulic Elevators.pdf: object of type 'generator' has no len()


📄 Document Information

📋 Submittal and Product Description_redacted.pdf
------------------------------------------------------------
   File size: 9.17 MB
   Pages: 35
   PDF version: PDF 1.7
   Creator: 
   Producer: 
   Document type: Submittal Document
   First page: 612 x 792 points
   Annotations on first page: 0

📋 Architectural Drawings_redacted.pdf
------------------------------------------------------------
   File size: 2.20 MB
   Pages: 8
   PDF version: PDF 1.4
   Creator: 
   Producer: 
   Document type: Architectural Drawing
   First page: 3024 x 2160 points
   Page rotation: 90°
   Annotations on first page: 0

📋 Spec 14 24 00 - Hydraulic Elevators.pdf
------------------------------------------------------------
   File size: 0.09 MB
   Pages: 6
   PDF version: PDF 1.7
   Creator: Bluebeam Revu x64
   Producer: Bluebeam PDF Library 21
   Document type: CSI Specification
   First page: 612 x 792 points
   Annotations on first page: 0

✅ Loaded information for 3 documents

### Document Parsing with Docling

In [10]:
def parse_document_with_docling(pdf_path: Path, document_name: str) -> Dict[str, Any]:
    """Parse a single PDF document using Docling"""
    print(f"🔄 Parsing {document_name}...")
    start_time = datetime.now()
    
    try:
        # Convert document
        result = converter.convert(str(pdf_path))
        doc = result.document
        
        # Extract basic document information
        doc_info = {
            "filename": pdf_path.name,
            "document_name": document_name,
            "parse_timestamp": start_time.isoformat(),
            "processing_time_seconds": (datetime.now() - start_time).total_seconds(),
            "success": True,
            "error": None
        }
        
        # Extract text content
        doc_info["full_text"] = doc.export_to_markdown()
        
        # Extract document structure
        doc_info["structure"] = []
        if hasattr(doc, 'chunks'):
            for chunk in doc.chunks:
                chunk_info = {
                    "type": chunk.label,
                    "text": chunk.text[:200] + "..." if len(chunk.text) > 200 else chunk.text,
                    "bbox": chunk.bbox if hasattr(chunk, 'bbox') else None
                }
                doc_info["structure"].append(chunk_info)
        
        # Extract tables
        doc_info["tables"] = []
        if hasattr(doc, 'tables'):
            for i, table in enumerate(doc.tables):
                table_info = {
                    "table_id": i,
                    "bbox": table.bbox if hasattr(table, 'bbox') else None,
                    "row_count": len(table.cells) if hasattr(table, 'cells') else 0,
                    "preview": str(table)[:200] + "..." if len(str(table)) > 200 else str(table)
                }
                doc_info["tables"].append(table_info)
        
        # Extract figures/images
        doc_info["figures"] = []
        if hasattr(doc, 'figures'):
            for i, figure in enumerate(doc.figures):
                figure_info = {
                    "figure_id": i,
                    "bbox": figure.bbox if hasattr(figure, 'bbox') else None,
                    "caption": figure.caption if hasattr(figure, 'caption') else None
                }
                doc_info["figures"].append(figure_info)
        
        # Extract annotations (if available)
        doc_info["annotations"] = []
        if hasattr(doc, 'annotations'):
            for i, annotation in enumerate(doc.annotations):
                annotation_info = {
                    "annotation_id": i,
                    "type": annotation.label if hasattr(annotation, 'label') else "unknown",
                    "text": annotation.text if hasattr(annotation, 'text') else None,
                    "bbox": annotation.bbox if hasattr(annotation, 'bbox') else None
                }
                doc_info["annotations"].append(annotation_info)
        
        print(f"✅ Successfully parsed {document_name}")
        print(f"   - Processing time: {doc_info['processing_time_seconds']:.2f} seconds")
        print(f"   - Text length: {len(doc_info['full_text'])} characters")
        print(f"   - Structure elements: {len(doc_info['structure'])}")
        print(f"   - Tables: {len(doc_info['tables'])}")
        print(f"   - Figures: {len(doc_info['figures'])}")
        print(f"   - Annotations: {len(doc_info['annotations'])}")
        
        return doc_info
        
    except Exception as e:
        error_info = {
            "filename": pdf_path.name,
            "document_name": document_name,
            "parse_timestamp": start_time.isoformat(),
            "processing_time_seconds": (datetime.now() - start_time).total_seconds(),
            "success": False,
            "error": str(e),
            "full_text": None,
            "structure": [],
            "tables": [],
            "figures": [],
            "annotations": []
        }
        
        print(f"❌ Error parsing {document_name}: {e}")
        return error_info

# Parse all documents
parsed_documents = {}

if converter is not None and pdf_files:
    print("🚀 Starting document parsing...")
    print("=" * 80)
    
    for pdf_file in pdf_files:
        document_name = pdf_file.stem  # filename without extension
        parsed_doc = parse_document_with_docling(pdf_file, document_name)
        parsed_documents[document_name] = parsed_doc
        print()  # Add spacing between documents
    
    print(f"✅ Completed parsing {len(parsed_documents)} documents")
else:
    print("❌ Cannot parse documents. Make sure:")
    print("   1. Docling converter is initialized (run Cell 2)")
    print("   2. PDF files are available (run Cell 1)")


2025-10-16 16:15:46,390 - docling.datamodel.document - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-16 16:15:46,445 - docling.document_converter - INFO - Going to convert document batch...
2025-10-16 16:15:46,450 - docling.document_converter - INFO - Initializing pipeline for StandardPdfPipeline with options hash 3a03610e4e170986e7b45cca197514c8


🚀 Starting document parsing...
🔄 Parsing Submittal and Product Description_redacted...


/Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/.venv/lib/python3.13/site-packages/docling/models/easyocr_model.py:68: UserWarning: Deprecated field. Better to set the `accelerator_options.device` in `pipeline_options`. When `use_gpu and accelerator_options.device == AcceleratorDevice.CUDA` the GPU is used to run EasyOCR. Otherwise, EasyOCR runs in CPU.
  warnings.warn(
2025-10-16 16:15:47,137 - easyocr.easyocr - WARNING - Downloading recognition model, please wait. This may take several minutes depending upon your network connection.
2025-10-16 16:15:47,593 - easyocr.easyocr - INFO - Download complete.
2025-10-16 16:15:49,941 - docling.utils.accelerator_utils - INFO - Accelerator device: 'mps'
2025-10-16 16:15:51,087 - docling.utils.accelerator_utils - INFO - Accelerator device: 'mps'
2025-10-16 16:15:51,708 - docling.pipeline.base_pipeline - INFO - Processing document Submittal and Product Description_redacted.pdf
2025-10-16 16:16:55,374 - docling.document_converte

✅ Successfully parsed Submittal and Product Description_redacted
   - Processing time: 69.00 seconds
   - Text length: 46216 characters
   - Structure elements: 0
   - Tables: 7
   - Figures: 0
   - Annotations: 0

🔄 Parsing Architectural Drawings_redacted...


2025-10-16 16:18:40,590 - docling.document_converter - INFO - Finished converting document Architectural Drawings_redacted.pdf in 105.17 sec.
2025-10-16 16:18:40,719 - docling.datamodel.document - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-16 16:18:40,720 - docling.document_converter - INFO - Going to convert document batch...
2025-10-16 16:18:40,721 - docling.pipeline.base_pipeline - INFO - Processing document Spec 14 24 00 - Hydraulic Elevators.pdf


✅ Successfully parsed Architectural Drawings_redacted
   - Processing time: 105.17 seconds
   - Text length: 185901 characters
   - Structure elements: 0
   - Tables: 9
   - Figures: 0
   - Annotations: 0

🔄 Parsing Spec 14 24 00 - Hydraulic Elevators...


2025-10-16 16:18:44,497 - docling.document_converter - INFO - Finished converting document Spec 14 24 00 - Hydraulic Elevators.pdf in 3.78 sec.


✅ Successfully parsed Spec 14 24 00 - Hydraulic Elevators
   - Processing time: 3.78 seconds
   - Text length: 21998 characters
   - Structure elements: 0
   - Tables: 0
   - Figures: 0
   - Annotations: 0

✅ Completed parsing 3 documents


### Parsing Results Visualization and Export

In [ ]:
import json
from IPython.display import display, HTML, Markdown

def display_document_structure(doc_name: str, doc_info: Dict[str, Any]):
    """Display the structure of a parsed document"""
    if not doc_info.get('success', False):
        print(f"❌ Cannot display structure for {doc_name}: {doc_info.get('error')}")
        return
    
    print(f"📋 Document Structure: {doc_name}")
    print("-" * 60)
    
    structure = doc_info.get('structure', [])
    if structure:
        for i, chunk in enumerate(structure[:10]):  # Show first 10 elements
            chunk_type = chunk.get('type', 'unknown')
            text_preview = chunk.get('text', '')[:100]
            print(f"{i+1:2d}. [{chunk_type:15s}] {text_preview}")
        
        if len(structure) > 10:
            print(f"    ... and {len(structure) - 10} more elements")
    else:
        print("No structure elements found")
    print()

def display_text_preview(doc_name: str, doc_info: Dict[str, Any], max_chars: int = 500):
    """Display a preview of the extracted text"""
    if not doc_info.get('success', False):
        print(f"❌ Cannot display text for {doc_name}: {doc_info.get('error')}")
        return
    
    print(f"📝 Text Preview: {doc_name}")
    print("-" * 60)
    
    full_text = doc_info.get('full_text', '')
    if full_text:
        preview = full_text[:max_chars]
        print(preview)
        if len(full_text) > max_chars:
            print(f"\n... ({len(full_text) - max_chars:,} more characters)")
    else:
        print("No text content extracted")
    print()

def save_parsing_results(parsed_documents: Dict[str, Dict[str, Any]], output_dir: Path):
    """Save parsing results to JSON files"""
    output_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save individual documents
    for doc_name, doc_info in parsed_documents.items():
        filename = f"{doc_name}_parsed_{timestamp}.json"
        filepath = output_dir / filename
        
        # Remove full_text from individual files to keep them manageable
        save_info = doc_info.copy()
        if 'full_text' in save_info:
            save_info['text_length'] = len(save_info['full_text'])
            del save_info['full_text']
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(save_info, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Saved {doc_name} results to {filepath}")
    
    # Save summary file
    summary_file = output_dir / f"parsing_summary_{timestamp}.json"
    summary = {
        "timestamp": timestamp,
        "total_documents": len(parsed_documents),
        "successful_documents": sum(1 for doc in parsed_documents.values() if doc.get('success', False)),
        "document_names": list(parsed_documents.keys()),
        "processing_times": {
            doc_name: doc_info.get('processing_time_seconds', 0) 
            for doc_name, doc_info in parsed_documents.items()
        }
    }
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Saved summary to {summary_file}")

# Display parsing results
if parsed_documents:
    # Show structure for first successful document
    for doc_name, doc_info in parsed_documents.items():
        if doc_info.get('success', False):
            display_document_structure(doc_name, doc_info)
            display_text_preview(doc_name, doc_info)
            break
    
    # Save results to JSON files
    output_dir = DATA_DIR / "parsing_results"
    save_parsing_results(parsed_documents, output_dir)
    
    print("✅ Parsing results displayed and saved")
else:
    print("❌ No parsed documents to display. Run Cell 4 first.")


📊 Parsing Results Summary
Total documents: 3
Successfully parsed: 3
Failed: 0

✅ Submittal and Product Description_redacted
   Processing time: 69.00s
   Text length: 46,216 chars
   Structure elements: 0
   Tables: 7
   Figures: 0
   Annotations: 0

✅ Architectural Drawings_redacted
   Processing time: 105.17s
   Text length: 185,901 chars
   Structure elements: 0
   Tables: 9
   Figures: 0
   Annotations: 0

✅ Spec 14 24 00 - Hydraulic Elevators
   Processing time: 3.78s
   Text length: 21,998 chars
   Structure elements: 0
   Tables: 0
   Figures: 0
   Annotations: 0

📋 Document Structure: Submittal and Product Description_redacted
------------------------------------------------------------
No structure elements found

📝 Text Preview: Submittal and Product Description_redacted
------------------------------------------------------------
07/22/2025

## Schindler Elevator Corporation Elevator Power Data

Job Name: Test Job 2024

Unit(s):

01

123

Capacity: 3500  lbs

Dallas,

Speed:

## CSI Division Classification

This cell will contain the CSI (Construction Specifications Institute) division classification logic. Based on the parsed document content, we'll:

1. **Identify CSI Division Codes** - Extract codes like "14 24 00" from document content
2. **Classify Document Sections** - Map content to appropriate CSI divisions
3. **Validate CSI Format** - Ensure codes follow the standard format
4. **Extract Division Metadata** - Gather additional context about each division


In [12]:
import re
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass

@dataclass
class CSIDivision:
    """CSI Division data structure"""
    code: str  # e.g., "14 24 00"
    name: str  # e.g., "Hydraulic Elevators"
    category: str  # e.g., "Conveying Equipment"
    confidence: float  # 0.0 to 1.0
    source_locations: List[Dict[str, Any]]  # Where it was found in the document

class CSIDivisionClassifier:
    """Classify document sections by CSI division codes"""
    
    def __init__(self):
        # CSI MasterFormat divisions (partial list - can be expanded)
        self.csi_divisions = {
            "01 00 00": {"name": "General Requirements", "category": "General"},
            "03 00 00": {"name": "Concrete", "category": "Concrete & Masonry"},
            "05 00 00": {"name": "Metals", "category": "Metals"},
            "07 00 00": {"name": "Thermal and Moisture Protection", "category": "Thermal Protection"},
            "08 00 00": {"name": "Openings", "category": "Openings"},
            "09 00 00": {"name": "Finishes", "category": "Finishes"},
            "14 00 00": {"name": "Conveying Equipment", "category": "Conveying Equipment"},
            "14 24 00": {"name": "Hydraulic Elevators", "category": "Conveying Equipment"},
            "14 25 00": {"name": "Electric Elevators", "category": "Conveying Equipment"},
            "16 00 00": {"name": "Plumbing", "category": "Plumbing"},
            "17 00 00": {"name": "HVAC", "category": "HVAC"},
            "18 00 00": {"name": "Electrical", "category": "Electrical"},
        }
        
        # CSI pattern matching
        self.csi_patterns = [
            r'\b(\d{2})\s+(\d{2})\s+(\d{2})\b',  # "14 24 00"
            r'\b(\d{2})-(\d{2})-(\d{2})\b',      # "14-24-00"
            r'\b(\d{6})\b',                      # "142400"
            r'Section\s+(\d{2})\s+(\d{2})\s+(\d{2})',  # "Section 14 24 00"
            r'Division\s+(\d{2})',               # "Division 14"
            r'CSI\s+(\d{2})\s+(\d{2})\s+(\d{2})', # "CSI 14 24 00"
        ]
        
        # Document type indicators
        self.document_indicators = {
            "specification": ["spec", "specification", "specs"],
            "drawing": ["drawing", "plan", "detail", "elevation", "section"],
            "submittal": ["submittal", "sd", "product", "pd", "manufacturer", "catalog"]
        }
    
    def normalize_csi_code(self, match_groups: Tuple[str, ...]) -> str:
        """Normalize CSI code to standard format (## ## ##)"""
        if len(match_groups) == 3:
            return f"{match_groups[0]} {match_groups[1]} {match_groups[2]}"
        elif len(match_groups) == 1 and len(match_groups[0]) == 6:
            # Convert "142400" to "14 24 00"
            code = match_groups[0]
            return f"{code[0:2]} {code[2:4]} {code[4:6]}"
        elif len(match_groups) == 1:
            # Single division number, pad with zeros
            return f"{match_groups[0].zfill(2)} 00 00"
        return ""
    
    def extract_csi_codes(self, text: str) -> List[CSIDivision]:
        """Extract CSI division codes from text"""
        csi_divisions = []
        
        for pattern in self.csi_patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE)
            
            for match in matches:
                groups = match.groups()
                normalized_code = self.normalize_csi_code(groups)
                
                if normalized_code:
                    # Check if this is a known CSI division
                    division_info = self.csi_divisions.get(normalized_code)
                    if division_info:
                        # Calculate confidence based on context
                        confidence = self.calculate_confidence(text, match.start(), match.end())
                        
                        # Find source location
                        source_location = {
                            "text_snippet": text[max(0, match.start()-50):match.end()+50],
                            "position": match.start(),
                            "match_text": match.group()
                        }
                        
                        csi_division = CSIDivision(
                            code=normalized_code,
                            name=division_info["name"],
                            category=division_info["category"],
                            confidence=confidence,
                            source_locations=[source_location]
                        )
                        
                        # Avoid duplicates
                        if not any(existing.code == normalized_code for existing in csi_divisions):
                            csi_divisions.append(csi_division)
        
        return sorted(csi_divisions, key=lambda x: x.confidence, reverse=True)
    
    def calculate_confidence(self, text: str, start_pos: int, end_pos: int) -> float:
        """Calculate confidence score for CSI code match"""
        confidence = 0.5  # Base confidence
        
        # Get context around the match
        context_start = max(0, start_pos - 100)
        context_end = min(len(text), end_pos + 100)
        context = text[context_start:context_end].lower()
        
        # Boost confidence for specification-related keywords
        spec_keywords = ["specification", "section", "part", "requirement", "standard"]
        for keyword in spec_keywords:
            if keyword in context:
                confidence += 0.1
        
        # Boost confidence for construction-related keywords
        construction_keywords = ["construction", "building", "project", "install", "provide"]
        for keyword in construction_keywords:
            if keyword in context:
                confidence += 0.05
        
        # Boost confidence for CSI-specific keywords
        csi_keywords = ["csi", "division", "masterformat"]
        for keyword in csi_keywords:
            if keyword in context:
                confidence += 0.15
        
        return min(1.0, confidence)
    
    def classify_document_type(self, filename: str, text: str) -> str:
        """Classify document type based on filename and content"""
        filename_lower = filename.lower()
        text_lower = text.lower()
        
        # Check filename indicators
        for doc_type, indicators in self.document_indicators.items():
            if any(indicator in filename_lower for indicator in indicators):
                return doc_type
        
        # Check content indicators
        if any(word in text_lower for word in ["specification", "section", "part 1", "part 2", "part 3"]):
            return "specification"
        elif any(word in text_lower for word in ["drawing", "plan", "elevation", "detail"]):
            return "drawing"
        elif any(word in text_lower for word in ["submittal", "product data", "manufacturer"]):
            return "submittal"
        
        return "unknown"
    
    def classify_document(self, document_name: str, doc_info: Dict[str, Any]) -> Dict[str, Any]:
        """Classify a complete document"""
        if not doc_info.get('success', False):
            return {
                "document_name": document_name,
                "classification_success": False,
                "error": doc_info.get('error', 'Document parsing failed')
            }
        
        text = doc_info.get('full_text', '')
        filename = doc_info.get('filename', '')
        
        # Extract CSI codes
        csi_divisions = self.extract_csi_codes(text)
        
        # Classify document type
        document_type = self.classify_document_type(filename, text)
        
        # Calculate overall CSI confidence
        overall_confidence = max([div.confidence for div in csi_divisions]) if csi_divisions else 0.0
        
        return {
            "document_name": document_name,
            "filename": filename,
            "classification_success": True,
            "document_type": document_type,
            "csi_divisions": [
                {
                    "code": div.code,
                    "name": div.name,
                    "category": div.category,
                    "confidence": div.confidence,
                    "source_locations": div.source_locations
                }
                for div in csi_divisions
            ],
            "primary_csi_division": csi_divisions[0].code if csi_divisions else None,
            "overall_confidence": overall_confidence,
            "total_csi_matches": len(csi_divisions)
        }

def display_csi_classification_results(classification_results: Dict[str, Dict[str, Any]]):
    """Display CSI classification results in a readable format"""
    print("🏗️ CSI Division Classification Results")
    print("=" * 80)
    
    for doc_name, result in classification_results.items():
        print(f"\n📋 {doc_name}")
        print("-" * 60)
        
        if not result.get('classification_success', False):
            print(f"❌ Classification failed: {result.get('error', 'Unknown error')}")
            continue
        
        # Document type
        doc_type = result.get('document_type', 'unknown')
        print(f"📄 Document type: {doc_type.title()}")
        
        # Primary CSI division
        primary_division = result.get('primary_csi_division')
        if primary_division:
            print(f"🏢 Primary CSI Division: {primary_division}")
        
        # Overall confidence
        confidence = result.get('overall_confidence', 0.0)
        print(f"🎯 Overall confidence: {confidence:.2f}")
        
        # All CSI divisions found
        csi_divisions = result.get('csi_divisions', [])
        print(f"🔍 Found {len(csi_divisions)} CSI division(s):")
        
        for i, division in enumerate(csi_divisions, 1):
            print(f"   {i}. {division['code']} - {division['name']}")
            print(f"      Category: {division['category']}")
            print(f"      Confidence: {division['confidence']:.2f}")
            
            # Show source location
            if division.get('source_locations'):
                location = division['source_locations'][0]
                snippet = location.get('text_snippet', '')[:100]
                print(f"      Context: ...{snippet}...")
        print()

# Initialize CSI classifier
csi_classifier = CSIDivisionClassifier()

# Classify all parsed documents
if parsed_documents:
    print("🔄 Starting CSI Division Classification...")
    print("=" * 80)
    
    classification_results = {}
    
    for doc_name, doc_info in parsed_documents.items():
        print(f"🔍 Classifying {doc_name}...")
        result = csi_classifier.classify_document(doc_name, doc_info)
        classification_results[doc_name] = result
    
    # Display results
    display_csi_classification_results(classification_results)
    
    print("✅ CSI classification completed")
else:
    print("❌ No parsed documents available for classification. Run cells 1-4 first.")
    classification_results = {}


🔄 Starting CSI Division Classification...
🔍 Classifying Submittal and Product Description_redacted...
🔍 Classifying Architectural Drawings_redacted...
🔍 Classifying Spec 14 24 00 - Hydraulic Elevators...
🏗️ CSI Division Classification Results

📋 Submittal and Product Description_redacted
------------------------------------------------------------
📄 Document type: Submittal
🎯 Overall confidence: 0.00
🔍 Found 0 CSI division(s):


📋 Architectural Drawings_redacted
------------------------------------------------------------
📄 Document type: Drawing
🎯 Overall confidence: 0.00
🔍 Found 0 CSI division(s):


📋 Spec 14 24 00 - Hydraulic Elevators
------------------------------------------------------------
📄 Document type: Specification
🏢 Primary CSI Division: 14 24 00
🎯 Overall confidence: 0.60
🔍 Found 1 CSI division(s):
   1. 14 24 00 - Hydraulic Elevators
      Category: Conveying Equipment
      Confidence: 0.60
      Context: ..., under cover, and in a dry location.

## SECTION 14 24 00



In [14]:
classification_results["Spec 14 24 00 - Hydraulic Elevators"]

{'document_name': 'Spec 14 24 00 - Hydraulic Elevators',
 'filename': 'Spec 14 24 00 - Hydraulic Elevators.pdf',
 'classification_success': True,
 'document_type': 'specification',
 'csi_divisions': [{'code': '14 24 00',
   'name': 'Hydraulic Elevators',
   'category': 'Conveying Equipment',
   'confidence': 0.6,
   'source_locations': [{'text_snippet': ', under cover, and in a dry location.\n\n## SECTION 14 24 00\n\n## HYDRAULIC ELEVATORS\n\nWRA Architects, Inc.\n\n23',
     'position': 2921,
     'match_text': '14 24 00'}]}],
 'primary_csi_division': '14 24 00',
 'overall_confidence': 0.6,
 'total_csi_matches': 1}

### Fact Extraction

In [ ]:
# Cell 8: CSI Specification Fact Extraction (Focused)

import re
from typing import List, Dict, Any, Optional, Tuple, Union
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation

@dataclass
class CSIFact:
    """Structured fact extracted from CSI specification documents"""
    id: str
    csi_division: str
    part: Optional[str] = None
    section: Optional[str] = None
    section_title: Optional[str] = None
    subsection: Optional[str] = None
    specification_type: str = "requirement"  # requirement, option, reference, general
    attribute: Optional[str] = None
    operator: Optional[str] = None
    value: Optional[Union[int, float, str]] = None
    unit: Optional[str] = None
    original_text: str = ""
    context: str = ""
    confidence: float = 0.8
    referenced_standards: List[str] = None
    source: Dict[str, Any] = None

class CSISpecificationExtractor:
    """Extract structured facts from CSI specification documents using hierarchical structure"""
    
    def __init__(self):
        # CSI MasterFormat structure patterns - more comprehensive
        self.part_patterns = {
            'part_1': r'(?:PART\s+1|PART\s+I)\s*[:\.]?\s*(?:GENERAL|GENERAL\s+REQUIREMENTS)',
            'part_2': r'(?:PART\s+2|PART\s+II)\s*[:\.]?\s*(?:PRODUCTS|PRODUCT)',
            'part_3': r'(?:PART\s+3|PART\s+III)\s*[:\.]?\s*(?:EXECUTION|INSTALLATION)',
        }
        
        # Section hierarchy patterns - more flexible
        self.section_patterns = [
            # Main sections: 1.1, 2.1, 3.1, etc.
            r'(\d+)\.(\d+)\s+([A-Z][A-Za-z\s&,\-\(\)]+?)(?=\n|$)',
            # Subsections: 2.1.A, 2.1.B, etc.
            r'(\d+)\.(\d+)\.([A-Z])\s+(.+)',
            # Sub-subsections: 2.1.1, 2.1.2, etc.
            r'(\d+)\.(\d+)\.(\d+)\s+(.+)',
        ]
        
        # Specification sentence patterns - more comprehensive
        self.spec_sentence_patterns = [
            # Requirements
            r'[^.]*(?:shall|must|required|minimum|maximum|not less than|not more than)[^.]*\.',
            # Technical specifications
            r'[^.]*(?:capacity|speed|pressure|voltage|power|dimension|size|weight)[^.]*\.',
            # Standards and codes
            r'[^.]*(?:ASME|ANSI|ASTM|IEEE|NFPA|UL|CSA|ISO|IEC|Building Code)[^.]*\.',
            # Materials and equipment
            r'[^.]*(?:material|equipment|component|system|device)[^.]*\.',
        ]
        
        # Technical value patterns - more comprehensive
        self.value_patterns = [
            # Capacity/load specifications
            r'(?:capacity|rated load|maximum load|minimum load|load capacity|carrying capacity)\s*(?:shall be|is|must be|required to be|not less than|not more than|minimum|maximum)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/\s°]+)',
            
            # Speed specifications
            r'(?:speed|velocity|travel speed|operating speed|hoisting speed|rated speed)\s*(?:shall be|is|must be|required to be|not less than|not more than|minimum|maximum)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/\s°]+)',
            
            # Dimension specifications
            r'(?:width|height|length|depth|thickness|diameter|size|clearance|headroom|floor space)\s*(?:shall be|is|must be|required to be|not less than|not more than|minimum|maximum)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/\s°]+)',
            
            # Pressure specifications
            r'(?:pressure|psi|psig|bar|kpa|working pressure|system pressure)\s*(?:shall be|is|must be|required to be|not less than|not more than|minimum|maximum)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/\s°]+)',
            
            # Power specifications
            r'(?:power|hp|horsepower|watt|kilowatt|kw|electrical power|motor power)\s*(?:shall be|is|must be|required to be|not less than|not more than|minimum|maximum)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/\s°]+)',
            
            # Voltage specifications
            r'(?:voltage|volt|v|electrical voltage|supply voltage|operating voltage)\s*(?:shall be|is|must be|required to be|not less than|not more than|minimum|maximum)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/\s°]+)',
            
            # Current specifications
            r'(?:current|ampere|amp|a|electrical current|motor current)\s*(?:shall be|is|must be|required to be|not less than|not more than|minimum|maximum)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/\s°]+)',
        ]
        
        # Standards patterns - more comprehensive
        self.standards_patterns = [
            r'(?:ASME|ANSI|ASTM|IEEE|NFPA|UL|CSA|ISO|IEC)\s+[A-Z]?\d+(?:\.\d+)?(?:[A-Z]\d+)?(?:\-\d+)?',
            r'(?:Building Code|Fire Code|Electrical Code|Plumbing Code|Mechanical Code)',
            r'(?:OSHA|EPA|DOT|FM)\s+[A-Z]?\d+(?:\.\d+)?',
        ]
        
        # Unit normalization mapping
        self.unit_conversions = {
            # Length units
            'inch': 'in', 'inches': 'in', '"': 'in', 'in': 'in',
            'foot': 'ft', 'feet': 'ft', "'": 'ft', 'ft': 'ft',
            'yard': 'yd', 'yards': 'yd', 'yd': 'yd',
            'millimeter': 'mm', 'millimeters': 'mm', 'mm': 'mm',
            'centimeter': 'cm', 'centimeters': 'cm', 'cm': 'cm',
            'meter': 'm', 'meters': 'm', 'm': 'm',
            
            # Weight/Mass units
            'pound': 'lbs', 'pounds': 'lbs', 'lb': 'lbs', 'lbs': 'lbs',
            'kilogram': 'kg', 'kilograms': 'kg', 'kg': 'kg',
            'ton': 'ton', 'tons': 'ton', 'ton': 'ton',
            
            # Pressure units
            'pounds per square inch': 'psi', 'psi': 'psi', 'psig': 'psig',
            'kilopascal': 'kpa', 'kpa': 'kpa',
            'bar': 'bar', 'bars': 'bar',
            
            # Speed units
            'feet per minute': 'fpm', 'fpm': 'fpm',
            'meters per second': 'm/s', 'm/s': 'm/s',
            'miles per hour': 'mph', 'mph': 'mph',
            
            # Electrical units
            'volt': 'V', 'volts': 'V', 'v': 'V',
            'ampere': 'A', 'amperes': 'A', 'amp': 'A', 'amps': 'A',
            'watt': 'W', 'watts': 'W', 'w': 'W',
            'kilowatt': 'kW', 'kilowatts': 'kW', 'kw': 'kW',
            'horsepower': 'hp', 'hp': 'hp',
            
            # Temperature units
            'degree fahrenheit': '°F', 'fahrenheit': '°F', '°f': '°F',
            'degree celsius': '°C', 'celsius': '°C', '°c': '°C',
        }
    
    def parse_csi_structure(self, text: str) -> Dict[str, Any]:
        """Parse CSI document structure (Part 1/2/3) with better detection"""
        print("🔍 Parsing CSI document structure...")
        
        structure = {
            'part_1': {'start': -1, 'end': -1, 'content': '', 'title': ''},
            'part_2': {'start': -1, 'end': -1, 'content': '', 'title': ''},
            'part_3': {'start': -1, 'end': -1, 'content': '', 'title': ''},
        }
        
        # Find all part headers
        for part_name, pattern in self.part_patterns.items():
            match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
            if match:
                structure[part_name]['start'] = match.start()
                structure[part_name]['title'] = match.group(0).strip()
                print(f"   Found {part_name}: {match.group(0)}")
        
        # Determine section boundaries
        parts = [(name, info['start']) for name, info in structure.items() if info['start'] >= 0]
        if not parts:
            print("   ⚠️  No CSI parts found, treating entire document as content")
            structure['part_2']['start'] = 0
            structure['part_2']['end'] = len(text)
            structure['part_2']['content'] = text
            structure['part_2']['title'] = 'ENTIRE DOCUMENT'
            return structure
        
        parts.sort(key=lambda x: x[1])
        
        for i, (part_name, start) in enumerate(parts):
            end = parts[i + 1][1] if i + 1 < len(parts) else len(text)
            structure[part_name]['end'] = end
            structure[part_name]['content'] = text[start:end]
            print(f"   {part_name}: {len(structure[part_name]['content'])} characters")
        
        return structure
    
    def extract_sections(self, part_content: str, part_name: str) -> List[Dict[str, Any]]:
        """Extract section hierarchy from part content"""
        print(f"🔍 Extracting sections from {part_name}...")
        sections = []
        
        for pattern in self.section_patterns:
            matches = re.finditer(pattern, part_content, re.IGNORECASE | re.MULTILINE)
            for match in matches:
                groups = match.groups()
                section = {
                    'full_match': match.group(0),
                    'start': match.start(),
                    'end': match.end(),
                    'section_number': groups[0],
                    'subsection_number': groups[1],
                    'title': groups[2].strip() if len(groups) > 2 else '',
                    'subtitle': groups[3].strip() if len(groups) > 3 else None,
                    'part': part_name
                }
                sections.append(section)
                print(f"   Found section: {groups[0]}.{groups[1]} - {groups[2].strip()}")
        
        return sections
    
    def extract_specification_sentences(self, content: str, section_info: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Extract sentences containing specifications from content"""
        sentences = []
        
        # Split into sentences more carefully
        sentence_pattern = r'[^.!?]*[.!?]+(?=\s|$)'
        all_sentences = re.findall(sentence_pattern, content)
        
        for sentence in all_sentences:
            sentence = sentence.strip()
            if len(sentence) < 30:  # Skip very short sentences
                continue
                
            # Check if sentence contains specification keywords
            for pattern in self.spec_sentence_patterns:
                if re.search(pattern, sentence, re.IGNORECASE):
                    sentences.append({
                        'text': sentence,
                        'specification_type': self.classify_specification_type(sentence),
                        'contains_values': bool(re.search(r'\d+', sentence)),
                        'contains_standards': bool(re.search(r'(?:ASME|ANSI|ASTM|IEEE|NFPA|UL|Building Code)', sentence, re.IGNORECASE)),
                        'section_info': section_info
                    })
                    break
        
        return sentences
    
    def classify_specification_type(self, sentence: str) -> str:
        """Classify specification sentence type"""
        sentence_lower = sentence.lower()
        
        if any(word in sentence_lower for word in ['shall', 'must', 'required', 'minimum', 'maximum']):
            return 'requirement'
        elif any(word in sentence_lower for word in ['may', 'can', 'option', 'optional']):
            return 'option'
        elif any(word in sentence_lower for word in ['refer to', 'see', 'according to', 'per', 'in accordance with']):
            return 'reference'
        elif any(word in sentence_lower for word in ['general', 'overview', 'description']):
            return 'general'
        else:
            return 'specification'
    
    def parse_technical_values(self, sentence: str) -> List[Dict[str, Any]]:
        """Parse technical values from specification sentences"""
        values = []
        
        for pattern in self.value_patterns:
            matches = re.finditer(pattern, sentence, re.IGNORECASE)
            for match in matches:
                operator_text = match.group(1) if match.group(1) else ""
                value_text = match.group(2)
                unit_text = match.group(3) if match.group(3) else ""
                
                # Parse operator
                operator = self.parse_operator(operator_text)
                
                # Parse value
                value = self.parse_value(value_text)
                
                # Normalize unit
                unit = self.normalize_unit(unit_text)
                
                # Extract attribute
                attribute = self.extract_attribute_from_sentence(sentence)
                
                values.append({
                    'operator': operator,
                    'value': value,
                    'unit': unit,
                    'attribute': attribute,
                    'original_match': match.group(0)
                })
        
        return values
    
    def parse_operator(self, operator_text: str) -> str:
        """Parse comparison operator"""
        if not operator_text:
            return "="
        
        operator_lower = operator_text.lower().strip()
        operator_map = {
            '>=': '>=', '<=': '<=', '>': '>', '<': '<',
            'not less than': '>=', 'not more than': '<=',
            'minimum': '>=', 'maximum': '<=',
            'at least': '>=', 'at most': '<=',
        }
        
        return operator_map.get(operator_lower, "=")
    
    def parse_value(self, value_text: str) -> Union[int, float, str]:
        """Parse numerical value"""
        if not value_text:
            return ""
        
        clean_value = value_text.replace(',', '').strip()
        
        try:
            decimal_value = Decimal(clean_value)
            return int(decimal_value) if decimal_value % 1 == 0 else float(decimal_value)
        except (InvalidOperation, ValueError):
            return clean_value
    
    def normalize_unit(self, unit_text: str) -> str:
        """Normalize unit text"""
        if not unit_text:
            return ""
        
        unit_lower = unit_text.lower().strip()
        return self.unit_conversions.get(unit_lower, unit_lower)
    
    def extract_attribute_from_sentence(self, sentence: str) -> str:
        """Extract attribute name from sentence"""
        sentence_lower = sentence.lower()
        
        attribute_map = {
            'capacity': 'capacity',
            'rated load': 'capacity',
            'maximum load': 'max_capacity',
            'minimum load': 'min_capacity',
            'load capacity': 'capacity',
            'carrying capacity': 'capacity',
            'speed': 'speed',
            'travel speed': 'speed',
            'operating speed': 'speed',
            'hoisting speed': 'speed',
            'rated speed': 'speed',
            'velocity': 'speed',
            'pressure': 'pressure',
            'working pressure': 'pressure',
            'system pressure': 'pressure',
            'voltage': 'voltage',
            'supply voltage': 'voltage',
            'operating voltage': 'voltage',
            'electrical voltage': 'voltage',
            'power': 'power',
            'electrical power': 'power',
            'motor power': 'power',
            'horsepower': 'power',
            'current': 'current',
            'electrical current': 'current',
            'motor current': 'current',
            'width': 'width',
            'height': 'height',
            'length': 'length',
            'depth': 'depth',
            'thickness': 'thickness',
            'diameter': 'diameter',
            'size': 'size',
            'clearance': 'clearance',
            'headroom': 'headroom',
            'floor space': 'floor_space',
        }
        
        for key, attribute in attribute_map.items():
            if key in sentence_lower:
                return attribute
        
        return 'specification_value'
    
    def extract_standards(self, content: str) -> List[str]:
        """Extract referenced standards"""
        standards = []
        
        for pattern in self.standards_patterns:
            matches = re.findall(pattern, content, re.IGNORECASE)
            standards.extend(matches)
        
        return list(set(standards))
    
    def extract_facts_from_document(self, document_name: str, doc_info: Dict[str, Any]) -> Dict[str, Any]:
        """Extract all facts from CSI specification document"""
        if not doc_info.get('success', False):
            return {
                "document_name": document_name,
                "extraction_success": False,
                "error": doc_info.get('error', 'Document parsing failed')
            }
        
        text = doc_info.get('full_text', '')
        print(f"🔄 Processing {document_name} ({len(text)} characters)")
        
        # Parse CSI document structure
        structure = self.parse_csi_structure(text)
        
        # Extract CSI division from document name
        csi_division = "14 24 00"  # Default from document name
        csi_match = re.search(r'(\d+\s+\d+\s+\d+)', document_name)
        if csi_match:
            csi_division = csi_match.group(1)
        
        facts = []
        all_sections = []
        
        # Process each part
        for part_name, part_info in structure.items():
            if part_info['start'] < 0 or not part_info['content']:
                continue
                
            print(f"\n📋 Processing {part_name}: {part_info['title']}")
            
            # Extract sections from this part
            sections = self.extract_sections(part_info['content'], part_name)
            all_sections.extend(sections)
            
            # Extract specification sentences from entire part content
            sentences = self.extract_specification_sentences(part_info['content'], {
                'part': part_name,
                'part_title': part_info['title']
            })
            
            # Extract standards from this part
            standards = self.extract_standards(part_info['content'])
            
            print(f"   Found {len(sentences)} specification sentences")
            print(f"   Found {len(standards)} referenced standards")
            
            # Create facts from sentences
            for sentence_info in sentences:
                sentence = sentence_info['text']
                
                # Parse technical values
                values = self.parse_technical_values(sentence)
                
                if values:
                    for value_info in values:
                        fact_id = f"csi_{hash(sentence) % 100000:05d}"
                        
                        fact = CSIFact(
                            id=fact_id,
                            csi_division=csi_division,
                            part=part_info['title'],
                            section=sentence_info['section_info'].get('section_number', '') + '.' + sentence_info['section_info'].get('subsection_number', ''),
                            section_title=sentence_info['section_info'].get('title', ''),
                            subsection=sentence_info['section_info'].get('subtitle'),
                            specification_type=sentence_info['specification_type'],
                            attribute=value_info['attribute'],
                            operator=value_info['operator'],
                            value=value_info['value'],
                            unit=value_info['unit'],
                            original_text=sentence,
                            context=sentence[:150] + "..." if len(sentence) > 150 else sentence,
                            confidence=0.8,
                            referenced_standards=standards,
                            source={
                                'document': doc_info.get('filename', ''),
                                'document_name': document_name,
                                'part': part_info['title'],
                                'csi_division': csi_division
                            }
                        )
                        
                        facts.append(fact)
                else:
                    # Even if no technical values, create a fact for important sentences
                    if sentence_info['specification_type'] in ['requirement', 'reference']:
                        fact_id = f"csi_{hash(sentence) % 100000:05d}"
                        
                        fact = CSIFact(
                            id=fact_id,
                            csi_division=csi_division,
                            part=part_info['title'],
                            section=sentence_info['section_info'].get('section_number', '') + '.' + sentence_info['section_info'].get('subsection_number', ''),
                            section_title=sentence_info['section_info'].get('title', ''),
                            subsection=sentence_info['section_info'].get('subtitle'),
                            specification_type=sentence_info['specification_type'],
                            attribute='general_requirement',
                            operator='=',
                            value=sentence,
                            unit=None,
                            original_text=sentence,
                            context=sentence[:150] + "..." if len(sentence) > 150 else sentence,
                            confidence=0.7,
                            referenced_standards=standards,
                            source={
                                'document': doc_info.get('filename', ''),
                                'document_name': document_name,
                                'part': part_info['title'],
                                'csi_division': csi_division
                            }
                        )
                        
                        facts.append(fact)
        
        # Convert facts to dictionaries for JSON serialization
        facts_dict = []
        for fact in facts:
            fact_dict = {
                'id': fact.id,
                'csi_division': fact.csi_division,
                'part': fact.part,
                'section': fact.section,
                'section_title': fact.section_title,
                'subsection': fact.subsection,
                'specification_type': fact.specification_type,
                'attribute': fact.attribute,
                'operator': fact.operator,
                'value': fact.value,
                'unit': fact.unit,
                'original_text': fact.original_text,
                'context': fact.context,
                'confidence': fact.confidence,
                'referenced_standards': fact.referenced_standards or [],
                'source': fact.source
            }
            facts_dict.append(fact_dict)
        
        return {
            "document_name": document_name,
            "filename": doc_info.get('filename', ''),
            "extraction_success": True,
            "facts": facts_dict,
            "total_facts": len(facts_dict),
            "csi_division": csi_division,
            "sections_found": len(all_sections),
            "parts_processed": len([p for p in structure.values() if p['start'] >= 0]),
            "high_confidence_facts": len([f for f in facts_dict if f['confidence'] >= 0.7]),
            "medium_confidence_facts": len([f for f in facts_dict if 0.5 <= f['confidence'] < 0.7]),
            "low_confidence_facts": len([f for f in facts_dict if f['confidence'] < 0.5])
        }

def display_csi_extraction_results(extraction_results: Dict[str, Dict[str, Any]]):
    """Display CSI specification extraction results"""
    print("🔍 CSI Specification Fact Extraction Results")
    print("=" * 80)
    
    for doc_name, result in extraction_results.items():
        print(f"\n📋 {doc_name}")
        print("-" * 60)
        
        if not result.get('extraction_success', False):
            print(f"❌ Extraction failed: {result.get('error', 'Unknown error')}")
            continue
        
        # Summary statistics
        total_facts = result.get('total_facts', 0)
        high_conf = result.get('high_confidence_facts', 0)
        medium_conf = result.get('medium_confidence_facts', 0)
        low_conf = result.get('low_confidence_facts', 0)
        sections_found = result.get('sections_found', 0)
        parts_processed = result.get('parts_processed', 0)
        
        print(f"📊 CSI Division: {result.get('csi_division', 'Unknown')}")
        print(f"📊 Parts processed: {parts_processed}")
        print(f"📊 Sections found: {sections_found}")
        print(f"📊 Total facts extracted: {total_facts}")
        print(f"   🟢 High confidence (≥0.7): {high_conf}")
        print(f"   🟡 Medium confidence (0.5-0.7): {medium_conf}")
        print(f"   🔴 Low confidence (<0.5): {low_conf}")
        
        # Show facts by type
        facts = result.get('facts', [])
        if facts:
            print(f"\n🎯 Extracted Facts:")
            
            # Group facts by specification type
            by_type = {}
            for fact in facts:
                spec_type = fact.get('specification_type', 'unknown')
                if spec_type not in by_type:
                    by_type[spec_type] = []
                by_type[spec_type].append(fact)
            
            for spec_type, type_facts in by_type.items():
                print(f"\n   📌 {spec_type.upper()} ({len(type_facts)} facts):")
                for i, fact in enumerate(type_facts[:5], 1):  # Show first 5 of each type
                    if fact.get('attribute') == 'general_requirement':
                        print(f"      {i}. {fact['value'][:80]}...")
                    else:
                        print(f"      {i}. {fact['attribute']} {fact['operator']} {fact['value']} {fact['unit'] or ''}")
                    print(f"         Part: {fact.get('part', 'N/A')}, Section: {fact.get('section', 'N/A')}")
                    if fact.get('referenced_standards'):
                        print(f"         Standards: {', '.join(fact['referenced_standards'][:2])}")
                    print(f"         Confidence: {fact['confidence']:.2f}")
                    print()
                
                if len(type_facts) > 5:
                    print(f"      ... and {len(type_facts) - 5} more {spec_type} facts")
        
        print()

# Initialize CSI specification extractor
csi_extractor = CSISpecificationExtractor()

# Extract facts from CSI specification documents only
if parsed_documents:
    print("🔄 Starting CSI Specification Fact Extraction...")
    print("=" * 80)
    
    extraction_results = {}
    
    for doc_name, doc_info in parsed_documents.items():
        # Only process specification documents
        classification = classification_results.get(doc_name, {})
        if classification.get('document_type') == 'specification':
            print(f"🔍 Extracting facts from {doc_name}...")
            result = csi_extractor.extract_facts_from_document(doc_name, doc_info)
            extraction_results[doc_name] = result
        else:
            print(f"⏭️  Skipping {doc_name} (not a specification document)")
    
    # Display results
    display_csi_extraction_results(extraction_results)
    
    print("✅ CSI specification fact extraction completed")
else:
    print("❌ No parsed documents available for fact extraction. Run cells 1-6 first.")
    extraction_results = {}


🔄 Starting Document-Type-Specific Fact Extraction...
🔍 Extracting facts from Submittal and Product Description_redacted (submittal)...
🔍 Extracting facts from Architectural Drawings_redacted (drawing)...
🔍 Extracting facts from Spec 14 24 00 - Hydraulic Elevators (specification)...
🔍 Document-Type-Specific Fact Extraction Results

📋 SUBMITTAL DOCUMENTS

📄 Submittal and Product Description_redacted
----------------------------------------
📊 Total facts: 2
   🟢 High confidence (≥0.7): 2
   🟡 Medium confidence (0.5-0.7): 0
   🔴 Low confidence (<0.5): 0

🎯 Extracted Facts:
   1. [Submittal] capacity: 3500 lbs
      Manufacturer: s Fuse Chart and rating verifying the SCCR meets requirements.

Per NFPA
      Confidence: 0.85

   2. [Submittal] speed: 100 fpm
      Manufacturer: s Fuse Chart and rating verifying the SCCR meets requirements.

Per NFPA
      Confidence: 0.85



📋 DRAWING DOCUMENTS

📄 Architectural Drawings_redacted
----------------------------------------
📊 Total facts: 11
   🟢

In [15]:
import re
from typing import List, Dict, Any, Optional, Tuple, Union
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation

@dataclass
class Fact:
    """Structured fact extracted from construction documents"""
    id: str
    topic: str
    attribute: str
    operator: str
    value: Union[int, float, str]
    unit: Optional[str]
    original_value: str
    confidence: float
    source: Dict[str, Any]
    context: str
    csi_division: Optional[str] = None
    section_id: Optional[str] = None
    subsection_id: Optional[str] = None
    manufacturer: Optional[str] = None
    product_code: Optional[str] = None
    model_number: Optional[str] = None
    standards: List[str] = None
    certifications: List[str] = None

class FactExtractor:
    """Extract structured facts from construction documents"""
    
    def __init__(self):
        # Unit conversion mappings
        self.unit_conversions = {
            # Length units
            'inch': 'in', 'inches': 'in', '"': 'in',
            'foot': 'ft', 'feet': 'ft', "'": 'ft',
            'yard': 'yd', 'yards': 'yd',
            'millimeter': 'mm', 'millimeters': 'mm',
            'centimeter': 'cm', 'centimeters': 'cm',
            'meter': 'm', 'meters': 'm',
            
            # Weight/Mass units
            'pound': 'lbs', 'pounds': 'lbs', 'lb': 'lbs',
            'kilogram': 'kg', 'kilograms': 'kg',
            'ton': 'ton', 'tons': 'ton',
            
            # Pressure units
            'pounds per square inch': 'psi', 'psi': 'psi',
            'kilopascal': 'kpa', 'kpa': 'kpa',
            'bar': 'bar', 'bars': 'bar',
            
            # Speed units
            'feet per minute': 'fpm', 'fpm': 'fpm',
            'meters per second': 'm/s', 'm/s': 'm/s',
            'miles per hour': 'mph', 'mph': 'mph',
            
            # Electrical units
            'volt': 'V', 'volts': 'V', 'v': 'V',
            'ampere': 'A', 'amperes': 'A', 'amp': 'A',
            'watt': 'W', 'watts': 'W', 'w': 'W',
            'horsepower': 'hp', 'hp': 'hp',
            
            # Temperature units
            'degree fahrenheit': '°F', 'fahrenheit': '°F', '°f': '°F',
            'degree celsius': '°C', 'celsius': '°C', '°c': '°C',
        }
        
        # Common construction topics
        self.topic_patterns = {
            'elevator': [r'elevator', r'lift', r'hoist'],
            'hydraulic_elevator': [r'hydraulic elevator', r'hydraulic lift'],
            'electric_elevator': [r'electric elevator', r'traction elevator'],
            'escalator': [r'escalator', r'moving stair'],
            'conveyor': [r'conveyor', r'conveying'],
            'crane': [r'crane', r'hoist', r'lifting'],
            'pump': [r'pump', r'pumping'],
            'motor': [r'motor', r'drive'],
            'generator': [r'generator', r'gen'],
            'transformer': [r'transformer', r'transform'],
            'switchgear': [r'switchgear', r'switch gear'],
            'panel': [r'panel', r'electrical panel'],
            'conduit': [r'conduit', r'raceway'],
            'cable': [r'cable', r'wire', r'wiring'],
            'insulation': [r'insulation', r'thermal'],
            'fire_protection': [r'fire protection', r'fire suppression', r'sprinkler'],
            'hvac': [r'hvac', r'heating', r'cooling', r'ventilation', r'air conditioning'],
            'plumbing': [r'plumbing', r'water', r'sewer', r'drainage'],
            'structural': [r'structural', r'steel', r'concrete', r'beam', r'column'],
        }
        
        # Specification patterns
        self.spec_patterns = [
            # Capacity/load specifications
            r'(?:capacity|rated load|maximum load|minimum load|load capacity)\s*(?:shall be|is|must be|required to be)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/°\s]+)',
            
            # Speed specifications
            r'(?:speed|velocity|travel speed|operating speed)\s*(?:shall be|is|must be|required to be)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/°\s]+)',
            
            # Dimension specifications
            r'(?:width|height|length|depth|thickness|diameter|size)\s*(?:shall be|is|must be|required to be)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/°\s]+)',
            
            # Pressure specifications
            r'(?:pressure|psi|psig|bar|kpa)\s*(?:shall be|is|must be|required to be)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/°\s]+)',
            
            # Power specifications
            r'(?:power|hp|horsepower|watt|kilowatt|kw)\s*(?:shall be|is|must be|required to be)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/°\s]+)',
            
            # Voltage specifications
            r'(?:voltage|volt|v)\s*(?:shall be|is|must be|required to be)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/°\s]+)',
            
            # Current specifications
            r'(?:current|ampere|amp|a)\s*(?:shall be|is|must be|required to be)?\s*([<>]=?|not less than|not more than|minimum|maximum)?\s*([\d,]+\.?\d*)\s*([a-zA-Z/°\s]+)',
        ]
        
        # Standards and codes patterns
        self.standards_patterns = [
            r'(?:ASME|ANSI|ASTM|IEEE|NFPA|UL|CSA|ISO|IEC)\s+[A-Z]?\d+(?:\.\d+)?(?:[A-Z]\d+)?',
            r'(?:Building Code|Fire Code|Electrical Code|Plumbing Code)',
            r'(?:OSHA|EPA|DOT)\s+[A-Z]?\d+(?:\.\d+)?',
        ]
        
        # Manufacturer patterns
        self.manufacturer_patterns = [
            r'(?:manufactured by|manufacturer|make|brand)\s*:?\s*([A-Z][a-zA-Z\s&.,]+)',
            r'(?:Otis|Schindler|Kone|ThyssenKrupp|Mitsubishi|Fujitec|Hitachi|Hyundai)',
        ]
    
    def normalize_unit(self, unit_text: str) -> Optional[str]:
        """Normalize unit text to standard format"""
        if not unit_text:
            return None
        
        unit_lower = unit_text.lower().strip()
        return self.unit_conversions.get(unit_lower, unit_lower)
    
    def parse_operator(self, operator_text: str) -> str:
        """Parse and normalize operator text"""
        if not operator_text:
            return "="
        
        operator_lower = operator_text.lower().strip()
        operator_map = {
            '>=': '>=', '<=': '<=', '>': '>', '<': '<',
            'not less than': '>=', 'not more than': '<=',
            'minimum': '>=', 'maximum': '<=',
            'at least': '>=', 'at most': '<=',
            'greater than': '>', 'less than': '<',
            'equal to': '=', 'equals': '=',
        }
        
        return operator_map.get(operator_lower, "=")
    
    def parse_value(self, value_text: str) -> Union[int, float, str]:
        """Parse numerical value from text"""
        if not value_text:
            return ""
        
        # Remove commas and clean the text
        clean_value = value_text.replace(',', '').strip()
        
        try:
            # Try to parse as decimal first for precision
            decimal_value = Decimal(clean_value)
            # Convert to float if it's a whole number, otherwise keep as float
            if decimal_value % 1 == 0:
                return int(decimal_value)
            else:
                return float(decimal_value)
        except (InvalidOperation, ValueError):
            # Return as string if not numerical
            return clean_value
    
    def extract_topic(self, text: str) -> Optional[str]:
        """Extract topic from text context"""
        text_lower = text.lower()
        
        for topic, patterns in self.topic_patterns.items():
            for pattern in patterns:
                if re.search(pattern, text_lower):
                    return topic
        
        return None
    
    def extract_specification_facts(self, text: str, source_info: Dict[str, Any]) -> List[Fact]:
        """Extract specification facts from text"""
        facts = []
        
        for pattern in self.spec_patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE)
            
            for match in matches:
                operator_text = match.group(1) if match.group(1) else ""
                value_text = match.group(2)
                unit_text = match.group(3) if match.group(3) else ""
                
                # Parse components
                operator = self.parse_operator(operator_text)
                value = self.parse_value(value_text)
                unit = self.normalize_unit(unit_text)
                
                # Extract topic from context
                topic = self.extract_topic(text[max(0, match.start()-100):match.end()+100])
                
                # Determine attribute from the matched text
                attribute = self.extract_attribute(match.group(0))
                
                # Create fact ID
                fact_id = f"fact_{hash(match.group(0)) % 100000:05d}"
                
                # Calculate confidence
                confidence = self.calculate_fact_confidence(match, text)
                
                fact = Fact(
                    id=fact_id,
                    topic=topic or "construction_equipment",
                    attribute=attribute,
                    operator=operator,
                    value=value,
                    unit=unit,
                    original_value=match.group(0),
                    confidence=confidence,
                    source=source_info,
                    context=text[max(0, match.start()-50):match.end()+50],
                    csi_division=source_info.get('csi_division'),
                    section_id=source_info.get('section_id'),
                    subsection_id=source_info.get('subsection_id')
                )
                
                facts.append(fact)
        
        return facts
    
    def extract_attribute(self, matched_text: str) -> str:
        """Extract attribute name from matched specification text"""
        matched_lower = matched_text.lower()
        
        # Map common specification terms to attributes
        attribute_map = {
            'capacity': 'capacity',
            'rated load': 'capacity',
            'maximum load': 'max_capacity',
            'minimum load': 'min_capacity',
            'load capacity': 'capacity',
            'speed': 'speed',
            'velocity': 'speed',
            'travel speed': 'travel_speed',
            'operating speed': 'operating_speed',
            'width': 'width',
            'height': 'height',
            'length': 'length',
            'depth': 'depth',
            'thickness': 'thickness',
            'diameter': 'diameter',
            'size': 'size',
            'pressure': 'pressure',
            'psi': 'pressure',
            'power': 'power',
            'hp': 'power',
            'horsepower': 'power',
            'voltage': 'voltage',
            'volt': 'voltage',
            'current': 'current',
            'ampere': 'current',
            'amp': 'current',
        }
        
        for key, attribute in attribute_map.items():
            if key in matched_lower:
                return attribute
        
        return 'specification_value'
    
    def calculate_fact_confidence(self, match, text: str) -> float:
        """Calculate confidence score for extracted fact"""
        confidence = 0.5  # Base confidence
        
        # Get context around the match
        context_start = max(0, match.start() - 100)
        context_end = min(len(text), match.end() + 100)
        context = text[context_start:context_end].lower()
        
        # Boost confidence for specification keywords
        spec_keywords = ['shall', 'must', 'required', 'specification', 'standard']
        for keyword in spec_keywords:
            if keyword in context:
                confidence += 0.1
        
        # Boost confidence for numerical precision
        if isinstance(self.parse_value(match.group(2)), (int, float)):
            confidence += 0.1
        
        # Boost confidence for unit presence
        if match.group(3) and match.group(3).strip():
            confidence += 0.1
        
        # Boost confidence for clear operator
        if match.group(1) and match.group(1).strip():
            confidence += 0.05
        
        return min(1.0, confidence)
    
    def extract_standards(self, text: str) -> List[str]:
        """Extract referenced standards and codes"""
        standards = []
        
        for pattern in self.standards_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            standards.extend(matches)
        
        return list(set(standards))  # Remove duplicates
    
    def extract_manufacturer_info(self, text: str) -> Dict[str, Optional[str]]:
        """Extract manufacturer information"""
        manufacturer_info = {
            'manufacturer': None,
            'product_code': None,
            'model_number': None
        }
        
        # Extract manufacturer
        for pattern in self.manufacturer_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                manufacturer_info['manufacturer'] = match.group(1).strip()
                break
        
        # Extract model numbers (common patterns)
        model_patterns = [
            r'model\s*(?:number|no\.?|#)?\s*:?\s*([A-Z0-9\-]+)',
            r'part\s*(?:number|no\.?|#)?\s*:?\s*([A-Z0-9\-]+)',
            r'catalog\s*(?:number|no\.?|#)?\s*:?\s*([A-Z0-9\-]+)',
        ]
        
        for pattern in model_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                manufacturer_info['model_number'] = match.group(1).strip()
                break
        
        return manufacturer_info
    
    def extract_facts_from_document(self, document_name: str, doc_info: Dict[str, Any]) -> Dict[str, Any]:
        """Extract all facts from a document"""
        if not doc_info.get('success', False):
            return {
                "document_name": document_name,
                "extraction_success": False,
                "error": doc_info.get('error', 'Document parsing failed')
            }
        
        text = doc_info.get('full_text', '')
        filename = doc_info.get('filename', '')
        
        # Prepare source information
        source_info = {
            'document': filename,
            'document_name': document_name,
            'page': 1,  # Default page, could be extracted from doc_info
            'section': 'document',
        }
        
        # Extract specification facts
        facts = self.extract_specification_facts(text, source_info)
        
        # Extract standards
        standards = self.extract_standards(text)
        
        # Extract manufacturer information
        manufacturer_info = self.extract_manufacturer_info(text)
        
        # Enhance facts with additional information
        for fact in facts:
            fact.standards = standards
            fact.manufacturer = manufacturer_info.get('manufacturer')
            fact.product_code = manufacturer_info.get('product_code')
            fact.model_number = manufacturer_info.get('model_number')
        
        # Convert facts to dictionaries for JSON serialization
        facts_dict = []
        for fact in facts:
            fact_dict = {
                'id': fact.id,
                'topic': fact.topic,
                'attribute': fact.attribute,
                'operator': fact.operator,
                'value': fact.value,
                'unit': fact.unit,
                'original_value': fact.original_value,
                'confidence': fact.confidence,
                'source': fact.source,
                'context': fact.context,
                'csi_division': fact.csi_division,
                'section_id': fact.section_id,
                'subsection_id': fact.subsection_id,
                'manufacturer': fact.manufacturer,
                'product_code': fact.product_code,
                'model_number': fact.model_number,
                'standards': fact.standards or [],
                'certifications': fact.certifications or []
            }
            facts_dict.append(fact_dict)
        
        return {
            "document_name": document_name,
            "filename": filename,
            "extraction_success": True,
            "facts": facts_dict,
            "total_facts": len(facts_dict),
            "standards": standards,
            "manufacturer_info": manufacturer_info,
            "high_confidence_facts": len([f for f in facts_dict if f['confidence'] >= 0.7]),
            "medium_confidence_facts": len([f for f in facts_dict if 0.5 <= f['confidence'] < 0.7]),
            "low_confidence_facts": len([f for f in facts_dict if f['confidence'] < 0.5])
        }

def display_fact_extraction_results(extraction_results: Dict[str, Dict[str, Any]]):
    """Display fact extraction results in a readable format"""
    print("🔍 Fact Extraction Results")
    print("=" * 80)
    
    for doc_name, result in extraction_results.items():
        print(f"\n📋 {doc_name}")
        print("-" * 60)
        
        if not result.get('extraction_success', False):
            print(f"❌ Extraction failed: {result.get('error', 'Unknown error')}")
            continue
        
        # Summary statistics
        total_facts = result.get('total_facts', 0)
        high_conf = result.get('high_confidence_facts', 0)
        medium_conf = result.get('medium_confidence_facts', 0)
        low_conf = result.get('low_confidence_facts', 0)
        
        print(f"📊 Total facts extracted: {total_facts}")
        print(f"   🟢 High confidence (≥0.7): {high_conf}")
        print(f"   🟡 Medium confidence (0.5-0.7): {medium_conf}")
        print(f"   🔴 Low confidence (<0.5): {low_conf}")
        
        # Standards found
        standards = result.get('standards', [])
        if standards:
            print(f"📜 Standards referenced: {len(standards)}")
            for std in standards[:3]:  # Show first 3
                print(f"   - {std}")
            if len(standards) > 3:
                print(f"   ... and {len(standards) - 3} more")
        
        # Manufacturer info
        mfg_info = result.get('manufacturer_info', {})
        if mfg_info.get('manufacturer'):
            print(f"🏭 Manufacturer: {mfg_info['manufacturer']}")
        if mfg_info.get('model_number'):
            print(f"🔢 Model: {mfg_info['model_number']}")
        
        # Show high confidence facts
        facts = result.get('facts', [])
        high_conf_facts = [f for f in facts if f['confidence'] >= 0.7]
        
        if high_conf_facts:
            print(f"\n🎯 High Confidence Facts ({len(high_conf_facts)}):")
            for i, fact in enumerate(high_conf_facts[:5], 1):  # Show first 5
                print(f"   {i}. {fact['topic']}.{fact['attribute']} {fact['operator']} {fact['value']} {fact['unit'] or ''}")
                print(f"      Confidence: {fact['confidence']:.2f}")
                print(f"      Context: ...{fact['context'][:80]}...")
                print()
        
        print()

# Initialize fact extractor
fact_extractor = FactExtractor()

# Extract facts from all parsed documents
if parsed_documents:
    print("🔄 Starting Fact Extraction...")
    print("=" * 80)
    
    extraction_results = {}
    
    for doc_name, doc_info in parsed_documents.items():
        print(f"🔍 Extracting facts from {doc_name}...")
        result = fact_extractor.extract_facts_from_document(doc_name, doc_info)
        extraction_results[doc_name] = result
    
    # Display results
    display_fact_extraction_results(extraction_results)
    
    print("✅ Fact extraction completed")
else:
    print("❌ No parsed documents available for fact extraction. Run cells 1-4 first.")
    extraction_results = {}

🔄 Starting Fact Extraction...
🔍 Extracting facts from Submittal and Product Description_redacted...
🔍 Extracting facts from Architectural Drawings_redacted...
🔍 Extracting facts from Spec 14 24 00 - Hydraulic Elevators...
🔍 Fact Extraction Results

📋 Submittal and Product Description_redacted
------------------------------------------------------------
📊 Total facts extracted: 37
   🟢 High confidence (≥0.7): 30
   🟡 Medium confidence (0.5-0.7): 7
   🔴 Low confidence (<0.5): 0
📜 Standards referenced: 4
   - electrical code
   - NFPA 70
   - ISO 9001
   ... and 1 more
🏭 Manufacturer: s Fuse Chart and rating verifying the SCCR meets requirements.

Per NFPA
🔢 Model: y

🎯 High Confidence Facts (30):
   1. construction_equipment.speed = 26.2 amps
      Confidence: 0.70
      Context: ...Size 70 Amps
- System Running Current at Constant Speed 26.2 Amps
- Recommended ...

   2. elevator.speed = 0.5 mb/s per elevator connected to the communication system
      Confidence: 0.80
      Context: ..

## Next Phase: Passage Chunking and Indexing

This cell will contain the passage extraction and chunking logic to prepare documents for search and retrieval. We'll implement:

1. **Text Chunking** - Split documents into meaningful passages
2. **Metadata Preservation** - Maintain source information for each chunk
3. **Search Preparation** - Format passages for vector and keyword search
4. **Quality Validation** - Ensure chunks are appropriately sized and meaningful

### Expected Implementation:
- Semantic chunking based on document structure
- Overlap handling for context preservation
- Metadata extraction for each passage
- Chunk size optimization (target: 200-500 tokens)
- Source citation preservation

### Expected Passage Format:
```json
{
  "passage_id": "spec_14_24_00_chunk_001",
  "text": "Hydraulic elevators shall have a minimum capacity of 2,500 pounds...",
  "source": {
    "document": "Spec_14_24_00_Hydraulic_Elevators.pdf",
    "page": 2,
    "section": "Part 2, Section 2.1.A",
    "csi_division": "14 24 00"
  },
  "metadata": {
    "chunk_size": 347,
    "contains_specifications": true,
    "contains_measurements": true
  }
}
```

*Add your passage chunking code here when ready to implement this phase.*


## Development Notes and Next Steps

### Current Status
✅ **Completed Phases:**
- Environment setup and dependency management
- Docling configuration for construction documents
- PDF file loading and metadata extraction
- Document parsing with Docling
- Results visualization and JSON export

### Next Development Phases
🔄 **Ready to Implement:**
- CSI Division Classification (Cell 6)
- Fact Extraction (Cell 7)
- Passage Chunking (Cell 8)

### Usage Instructions

1. **Start Here**: Run cells 1-5 in sequence to parse your documents
2. **Inspect Results**: Review the parsing results in Cell 5
3. **Add Next Phase**: Implement CSI classification in Cell 6
4. **Iterate**: Continue adding cells for each processing phase
5. **Validate**: Check results at each step before proceeding

### Tips for Development

- **Run cells incrementally** - Don't run all cells at once
- **Check outputs** - Verify parsing results before adding new phases
- **Save frequently** - Export results to JSON for inspection
- **Debug issues** - Use the logging output to troubleshoot problems
- **Extend gradually** - Add complexity one feature at a time

### Troubleshooting

- **Docling errors**: Make sure all dependencies are installed
- **File not found**: Verify the `data/` directory contains your PDF files
- **Memory issues**: Process documents one at a time for large files
- **Parsing failures**: Check PDF format compatibility with Docling

### Integration with Backend

Once you've validated the processing pipeline in this notebook, you can:
1. Extract the working code into Python modules
2. Integrate with the FastAPI backend
3. Add database persistence
4. Implement the full agentic workflow
